# Chunking And Lexical Audit

Notebook de contrôle visuel des chunks et du retrieval lexical sur le subset `10`.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bofip_cleanroom.jsonio import read_jsonl
from bofip_cleanroom.lexical_retrieval import LexicalBM25Index
from bofip_cleanroom.models import chunk_node_from_dict

strategies = {
    'paragraph_preserving': PROJECT_ROOT / 'data' / 'interim' / 'chunks_paragraph_preserving_sample_10.jsonl',
    'section_window': PROJECT_ROOT / 'data' / 'interim' / 'chunks_section_window_sample_10.jsonl',
    'parent_child': PROJECT_ROOT / 'data' / 'interim' / 'chunks_parent_child_sample_10.jsonl',
}
reports = {
    name: json.loads((PROJECT_ROOT / 'data' / 'reports' / f'phase2_chunks_{name}_sample_10.json').read_text(encoding='utf-8'))
    for name in strategies
}
chunks_by_strategy = {
    name: [chunk_node_from_dict(item) for item in read_jsonl(path)]
    for name, path in strategies.items()
}
reports

{'paragraph_preserving': {'generated_at': '2026-04-11T17:09:05.085306+00:00',
  'strategy': 'paragraph_preserving',
  'document_count': 10,
  'chunk_count': 211,
  'chunk_kind_counts': {'paragraph': 211},
  'token_count_stats': {'count': 211, 'min': 1, 'max': 264, 'avg': 41.14},
  'too_long_count': 0,
  'empty_text_count': 0,
  'chunks_path': 'C:\\Users\\rapha\\Desktop\\Document perso\\Projet compta\\bofip-rag-cleanroom\\data\\interim\\chunks_paragraph_preserving_sample_10.jsonl'},
 'section_window': {'generated_at': '2026-04-11T17:09:05.123084+00:00',
  'strategy': 'section_window',
  'document_count': 10,
  'chunk_count': 52,
  'chunk_kind_counts': {'paragraph': 11, 'paragraph_window': 41},
  'token_count_stats': {'count': 52, 'min': 1, 'max': 348, 'avg': 166.96},
  'too_long_count': 0,
  'empty_text_count': 0,
  'chunks_path': 'C:\\Users\\rapha\\Desktop\\Document perso\\Projet compta\\bofip-rag-cleanroom\\data\\interim\\chunks_section_window_sample_10.jsonl'},
 'parent_child': {'gen

In [2]:
for name, chunks in chunks_by_strategy.items():
    print('\n===', name, '===')
    for chunk in chunks[:5]:
        print(chunk.chunk_kind, '|', chunk.boi_reference, '|', chunk.section_path, '|', repr(chunk.text[:180]))


=== paragraph_preserving ===
paragraph | BOI-ANNX-000098-20120912 | ["I. Zones sinistrées de l'Ariège"] | '- Cantons et communes de la zone nord du département, dite « Nord du Plantaurel » :'
paragraph | BOI-ANNX-000098-20120912 | ["I. Zones sinistrées de l'Ariège"] | "Cantons de : Le Fossat, Le Mas d'Azil, Mirepoix, Pamiers, Quérigut, Sainte-Croix-Volvestre, Saint-Lizier, Saverdun, Varilhes."
paragraph | BOI-ANNX-000098-20120912 | ["I. Zones sinistrées de l'Ariège"] | "Communes de : Baulou, Loubières, Vernajoul, Saint-Jean-de-Verges, L'Herm, Aigues-Juntes, Allières, Durban-sur-Arize, Montseron, Carla-de-Roquefort, Lieurac, Sautel, Clermont."
paragraph | BOI-ANNX-000098-20120912 | ["I. Zones sinistrées de l'Ariège"] | '- Cantons et communes de la zone sud du département, dite « Sud du Plantaurel » :'
paragraph | BOI-ANNX-000098-20120912 | ["I. Zones sinistrées de l'Ariège"] | 'Cantons de : Ax-les-Thermes, Castillon-en-Couserans, Les Cabannes, Massat, Oust, Tarascon-sur-Ariège, Vicdess

## Chunks courts

Repérer les fragments trop courts ou peu informatifs.

In [3]:
for name, chunks in chunks_by_strategy.items():
    print('\n===', name, 'shortest ===')
    for chunk in sorted(chunks, key=lambda c: (c.token_count, len(c.text)))[:12]:
        print(chunk.token_count, '|', chunk.boi_reference, '|', chunk.chunk_kind, '|', repr(chunk.text[:120]))


=== paragraph_preserving shortest ===
1 | BOI-ANNX-000098-20120912 | paragraph | 'Département.'
2 | BOI-BAREME-000024-20160706 | paragraph | 'Remarques :'
4 | BOI-ANNX-000098-20120912 | paragraph | 'Zone 1 :'
4 | BOI-ANNX-000098-20120912 | paragraph | 'Zone 1 :'
4 | BOI-ANNX-000098-20120912 | paragraph | 'Zone 2 :'
4 | BOI-ANNX-000098-20120912 | paragraph | '-Zone 1 :'
4 | BOI-ANNX-000098-20120912 | paragraph | '-Zone 2 :'
4 | BOI-ANNX-000098-20120912 | paragraph | '- Zone2 :'
4 | BOI-FORM-000098-20250723 | paragraph | 'Numéro SIREN :'
4 | BOI-ANNX-000098-20120912 | paragraph | 'Ensemble du département.'
5 | BOI-ANNX-000098-20120912 | paragraph | '- Zone 3 :'
5 | BOI-ANNX-000098-20120912 | paragraph | '- Zone 1 :'

=== section_window shortest ===
1 | BOI-ANNX-000098-20120912 | paragraph | 'Département.'
4 | BOI-ANNX-000098-20120912 | paragraph | 'Ensemble du département.'
8 | BOI-ANNX-000098-20120912 | paragraph | 'Cantons de : Bourg-de-Visa, Lauzerte, Montaigu-de-Quercy.'
19 | BOI-AN

## Queries smoke

Comparer les stratégies sur quelques requêtes représentatives.

In [4]:
queries = [
    'BOI-AIS-MOB-10 taxes sur les deplacements routiers',
    'taxe sur la mise en relation par voie electronique transport',
    'zones sinistrees secheresse 2009 ariege',
    'benefices agricoles forfaitaires cultures specialisees revenus 2015',
    'article 151 septies A cessions echelonnees',
    'modele option report d imposition article 151 octies A',
]

for name, chunks in chunks_by_strategy.items():
    index = LexicalBM25Index(chunks)
    print('\n==============================')
    print(name)
    print('==============================')
    for query in queries:
        print('\nQ:', query)
        for hit in index.search(query, top_k=3):
            print(f"  {hit.rank}. {hit.chunk.boi_reference} | {hit.chunk.chunk_kind} | {hit.score:.3f}")
            print('     ', ' > '.join(hit.chunk.section_path))
            print('     ', hit.chunk.text[:180].replace('\n', ' '))


paragraph_preserving

Q: BOI-AIS-MOB-10 taxes sur les deplacements routiers
  1. BOI-AIS-MOB-10-20240710 | paragraph | 28.922
      AIS - Mobilités - Taxes sur les déplacements routiers
      les taxes sur l'immatriculation des véhicules (chapitre 2, BOI-AIS-MOB-10-20 ) ;
  2. BOI-AIS-MOB-10-20240710 | paragraph | 28.886
      AIS - Mobilités - Taxes sur les déplacements routiers
      les dispositions générales (chapitre 1, BOI-AIS-MOB-10-10 ) ;
  3. BOI-AIS-MOB-10-20240710 | paragraph | 28.040
      AIS - Mobilités - Taxes sur les déplacements routiers
      les taxes sur l’affectation des véhicules à des fins économiques (chapitre 3, BOI-AIS-MOB-10-30 ) ;

Q: taxe sur la mise en relation par voie electronique transport
  1. BOI-AIS-CCN-30-40-20250409 | paragraph | 19.540
      AIS - Culture, communication et numérique - Taxes portant sur l’utilisation finale des réseaux de communications électroniques - Taxe sur la mise en relation par voie électronique en vue de fournir certaines 

## Batch results

Lire les scores du batch lexical reproductible.

In [5]:
batch_reports = {
    name: json.loads((PROJECT_ROOT / 'data' / 'reports' / f'phase3_batch_eval_chunks_{name}_sample_10.json').read_text(encoding='utf-8'))
    for name in strategies
}
batch_reports

{'paragraph_preserving': {'generated_at': '2026-04-11T17:12:27.139828+00:00',
  'chunks_path': 'C:\\Users\\rapha\\Desktop\\Document perso\\Projet compta\\bofip-rag-cleanroom\\data\\interim\\chunks_paragraph_preserving_sample_10.jsonl',
  'queries_path': 'C:\\Users\\rapha\\Desktop\\Document perso\\Projet compta\\bofip-rag-cleanroom\\data\\interim\\lexical_queries_sample_10.jsonl',
  'query_count': 6,
  'metrics': {'hit@1': 1.0, 'hit@3': 1.0, 'hit@5': 1.0},
  'results': [{'id': 'q1',
    'query': 'BOI-AIS-MOB-10 taxes sur les deplacements routiers',
    'expected_boi': 'BOI-AIS-MOB-10-20240710',
    'returned_boi': ['BOI-AIS-MOB-10-20240710',
     'BOI-AIS-MOB-10-20240710',
     'BOI-AIS-MOB-10-20240710',
     'BOI-AIS-MOB-10-20240710',
     'BOI-AIS-MOB-10-20240710'],
    'hit@1': True,
    'hit@3': True,
    'hit@5': True},
   {'id': 'q2',
    'query': 'taxe sur la mise en relation par voie electronique transport',
    'expected_boi': 'BOI-AIS-CCN-30-40-20250409',
    'returned_boi': [

## Lecture

- si tout est à `1.0`, le smoke set valide la base mais ne suffit pas à départager les stratégies
- le vrai travail suivant est d'ajouter des requêtes plus dures, des cas ambigus et des cas non supportés